# Ensemble NER Evaluation on MACCROBAT

This notebook evaluates the three-model ensemble in `ensemble_pretrained_ner.py` against MACCROBAT. It uses the ensemble's normalized meaning groups and exact character-span matching.

MACCROBAT documents are reconstructed by joining their provided tokens with spaces. Long documents are processed in non-overlapping word chunks so every model stays within its token limit. Predictions for PHI-only groups that MACCROBAT does not annotate are excluded from this scoped benchmark.

## Imports and Configuration

In [1]:
from __future__ import annotations

from collections import defaultdict
from dataclasses import replace
from pathlib import Path
import sys

import pandas as pd
from datasets import load_dataset
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent:
    ensemble_module = (
        PROJECT_ROOT
        / "models"
        / "pre_fine_tuned_models"
        / "ensemble_pretrained_ner.py"
    )
    if ensemble_module.exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent
else:
    raise FileNotFoundError("Could not locate the Deep-learning-Lab project root.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from models.pre_fine_tuned_models.ensemble_pretrained_ner import (
    MODEL_SPECS,
    load_pipeline,
    meaning_group_for,
    resolve_device,
    resolve_overlaps,
    run_model,
)

WORD_CHUNK_SIZE = 150
MAX_DOCUMENTS = None  # Use a small number such as 5 for a quick smoke test.
DEVICE = "auto"

## Load MACCROBAT and the Ensemble Models

All three models are loaded once and reused across documents. Running all 400 documents can take a long time on CPU.

In [2]:
maccrobat = load_dataset(
    "ktgiahieu/maccrobat2018_2020",
    split="train",
)

evaluation_dataset = (
    maccrobat
    if MAX_DOCUMENTS is None
    else maccrobat.select(range(min(MAX_DOCUMENTS, len(maccrobat))))
)

device = resolve_device(DEVICE)
model_pipelines = {
    spec.alias: load_pipeline(spec, device=device)
    for spec in MODEL_SPECS
}

print("Evaluation documents:", len(evaluation_dataset))
print("Models:", list(model_pipelines))
print("Device:", "cuda" if device == 0 else "cpu")

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Evaluation documents: 400
Models: ['biomedical_all', 'medication_ner', 'phi_deidentifier']
Device: cpu


## Build Text and Gold Entity Spans

In [3]:
def build_document(tokens: list[str]) -> tuple[str, list[int], list[int]]:
    starts = []
    ends = []
    cursor = 0

    for token in tokens:
        starts.append(cursor)
        cursor += len(token)
        ends.append(cursor)
        cursor += 1

    return " ".join(tokens), starts, ends


def build_gold_entities(tags, starts, ends):
    entities = []
    current_group = None
    current_start = None
    current_end = None

    def close_current():
        if current_group is not None:
            entities.append((current_group, current_start, current_end))

    for tag, token_start, token_end in zip(tags, starts, ends):
        if tag == "O":
            close_current()
            current_group = current_start = current_end = None
            continue

        bio_prefix = tag[0]
        meaning_group = meaning_group_for(tag)
        starts_new_entity = bio_prefix == "B" or meaning_group != current_group

        if starts_new_entity:
            close_current()
            current_group = meaning_group
            current_start = token_start
        current_end = token_end

    close_current()
    return entities


dataset_tags = {
    tag
    for document_tags in maccrobat["tags"]
    for tag in document_tags
    if tag != "O"
}
MACCROBAT_MEANING_GROUPS = {meaning_group_for(tag) for tag in dataset_tags}

print("MACCROBAT meaning groups:", len(MACCROBAT_MEANING_GROUPS))
print(sorted(MACCROBAT_MEANING_GROUPS))

MACCROBAT meaning groups: 38
['ACTIVITY', 'AGE', 'ANATOMY', 'AREA', 'BIOLOGICAL_ATTRIBUTE', 'CLINICAL_EVENT', 'CLINICAL_REASON_OR_CONDITION', 'COLOR', 'COREFERENCE', 'DATE', 'DETAILED_DESCRIPTION', 'DISTANCE', 'FAMILY_HISTORY', 'HEIGHT', 'HISTORY', 'LAB_VALUE', 'LOCATION_OR_CARE_SITE', 'MASS', 'MEDICATION', 'MEDICATION_ADMINISTRATION', 'MEDICATION_AMOUNT', 'MEDICATION_DURATION', 'MEDICATION_FREQUENCY', 'OCCUPATION', 'OTHER_ENTITY', 'OTHER_EVENT', 'OUTCOME', 'PERSONAL_BACKGROUND', 'PROCEDURE', 'QUALITATIVE_CONCEPT', 'QUANTITATIVE_CONCEPT', 'SEX_OR_GENDER_CUE', 'SHAPE', 'SUBJECT', 'TEXTURE', 'TIME', 'VOLUME', 'WEIGHT']


## Run the Ensemble on One Document

Each document is divided into word chunks for inference. Candidate offsets are shifted back into document coordinates before the ensemble resolves overlaps.

In [5]:
def run_ensemble_on_document(tokens: list[str]):
    document_text, starts, ends = build_document(tokens)
    all_candidates = []
    next_candidate_id = 1

    for first_word in range(0, len(tokens), WORD_CHUNK_SIZE):
        last_word = min(first_word + WORD_CHUNK_SIZE, len(tokens))
        chunk_start = starts[first_word]
        chunk_end = ends[last_word - 1]
        chunk_text = document_text[chunk_start:chunk_end]

        for spec in MODEL_SPECS:
            candidates, next_candidate_id = run_model(
                model_pipelines[spec.alias],
                spec,
                chunk_text,
                next_candidate_id,
            )
            all_candidates.extend(
                replace(
                    candidate,
                    start=candidate.start + chunk_start if candidate.start >= 0 else candidate.start,
                    end=candidate.end + chunk_start if candidate.end >= 0 else candidate.end,
                )
                for candidate in candidates
            )

    selected = resolve_overlaps(all_candidates)
    return [
        candidate
        for candidate in selected
        if candidate.meaning_group in MACCROBAT_MEANING_GROUPS
        and candidate.start >= 0
        and candidate.end > candidate.start
    ]

## Evaluate Exact Entity Spans

A prediction is correct only when its normalized meaning group, start offset, and end offset exactly match a gold entity.

In [6]:
group_counts = defaultdict(lambda: {"tp": 0, "fp": 0, "fn": 0})
document_rows = []

for document_index, row in enumerate(tqdm(evaluation_dataset, desc="Evaluating ensemble")):
    _, starts, ends = build_document(row["tokens"])
    gold_entities = set(build_gold_entities(row["tags"], starts, ends))
    selected = run_ensemble_on_document(row["tokens"])
    predicted_entities = {
        (candidate.meaning_group, candidate.start, candidate.end)
        for candidate in selected
    }

    true_positives = gold_entities & predicted_entities
    false_positives = predicted_entities - gold_entities
    false_negatives = gold_entities - predicted_entities

    for group, _, _ in true_positives:
        group_counts[group]["tp"] += 1
    for group, _, _ in false_positives:
        group_counts[group]["fp"] += 1
    for group, _, _ in false_negatives:
        group_counts[group]["fn"] += 1

    document_rows.append(
        {
            "document": document_index,
            "gold_entities": len(gold_entities),
            "predicted_entities": len(predicted_entities),
            "true_positives": len(true_positives),
            "false_positives": len(false_positives),
            "false_negatives": len(false_negatives),
        }
    )

document_results = pd.DataFrame(document_rows)
document_results.head()

Evaluating ensemble:   0%|          | 0/400 [00:00<?, ?it/s]

,document,gold_entities,predicted_entities,true_positives,false_positives,false_negatives
0,0,74,97,38,59,36
1,1,55,138,27,111,28
2,2,71,97,26,71,45
3,3,42,126,22,104,20
4,4,66,105,30,75,36


## Results

In [7]:
def safe_divide(numerator: int, denominator: int) -> float:
    return numerator / denominator if denominator else 0.0


result_rows = []
for group, counts in sorted(group_counts.items()):
    precision = safe_divide(counts["tp"], counts["tp"] + counts["fp"])
    recall = safe_divide(counts["tp"], counts["tp"] + counts["fn"])
    result_rows.append(
        {
            "meaning_group": group,
            **counts,
            "precision": precision,
            "recall": recall,
            "f1": safe_divide(2 * precision * recall, precision + recall),
        }
    )

per_group_results = pd.DataFrame(result_rows)

total_tp = sum(counts["tp"] for counts in group_counts.values())
total_fp = sum(counts["fp"] for counts in group_counts.values())
total_fn = sum(counts["fn"] for counts in group_counts.values())
micro_precision = safe_divide(total_tp, total_tp + total_fp)
micro_recall = safe_divide(total_tp, total_tp + total_fn)

summary = pd.Series(
    {
        "documents": len(evaluation_dataset),
        "true_positives": total_tp,
        "false_positives": total_fp,
        "false_negatives": total_fn,
        "precision": micro_precision,
        "recall": micro_recall,
        "f1": safe_divide(
            2 * micro_precision * micro_recall,
            micro_precision + micro_recall,
        ),
    },
    name="exact_span_micro_metrics",
)

display(summary)
per_group_results.sort_values("f1", ascending=False)

documents            400.000000
true_positives      9836.000000
false_positives    32772.000000
false_negatives    16850.000000
precision              0.230849
recall                 0.368583
f1                     0.283892
Name: exact_span_micro_metrics, dtype: float64

,meaning_group,tp,fp,fn,precision,recall,f1
1,AGE,388,16,12,0.960396,0.970000,0.965174
31,SEX_OR_GENDER_CUE,352,26,14,0.931217,0.961749,0.946237
27,PERSONAL_BACKGROUND,90,24,8,0.789474,0.918367,0.849057
26,OUTCOME,49,29,20,0.628205,0.710145,0.666667
23,OCCUPATION,16,6,10,0.727273,0.615385,0.666667
3,AREA,40,52,10,0.434783,0.800000,0.563380
5,CLINICAL_EVENT,474,440,341,0.518600,0.581595,0.548294
33,SUBJECT,36,36,30,0.500000,0.545455,0.521739
16,LOCATION_OR_CARE_SITE,273,301,232,0.475610,0.540594,0.506024
13,HEIGHT,4,4,4,0.500000,0.500000,0.500000
